2. Diagnóstico de qualidade.
2.1 - Dimensões, tipos, uso de memória
Aqui estamos analisando a estrutura do CSV, verificação estrutural e de performance (dimensões, tipos e uso de memória)

In [1]:
import pandas as pd

def analisar_estrutura(caminho_arquivo):
    print("--- 1. DIAGNÓSTICO DE ESTRUTURA E MEMÓRIA ---")
    df = pd.read_csv(caminho_arquivo)
    
    # Dimensões
    linhas, colunas = df.shape
    print(f"Dimensões: {linhas} linhas e {colunas} colunas")
    
    # Uso de memória (em Megabytes)
    memoria_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"Uso de Memória: {memoria_mb:.2f} MB\n")
    
    # Tipos de dados
    print("Tipos de dados por coluna:")
    print(df.dtypes)
    
    return df

# Execução
if __name__ == "__main__":
    df_base = analisar_estrutura("AirbnbRJRentals.csv")


--- 1. DIAGNÓSTICO DE ESTRUTURA E MEMÓRIA ---
Dimensões: 28301 linhas e 16 colunas
Uso de Memória: 11.19 MB

Tipos de dados por coluna:
id                                  int64
name                                  str
host_id                             int64
host_name                             str
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                           str
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
number_of_reviews_ltm               int64
dtype: object


2. Diagnóstico de qualidade
2.2 Faltantes por coluna (quantidade e percentual)
Aqui vemos quais dados faltam do CSV, verificando as quantidades e os percentuais

In [2]:
import pandas as pd

def analisar_faltantes(caminho_arquivo):
    print("\n--- 2. DIAGNÓSTICO DE DADOS FALTANTES ---")
    
    # 1. Carrega o CSV direto neste script
    df = pd.read_csv(caminho_arquivo)
    
    # 2. Cálculo de nulos e percentuais
    qtd_faltantes = df.isnull().sum()
    perc_faltantes = (df.isnull().sum() / len(df)) * 100
    
    # 3. Criação de uma tabela resumo
    tabela_faltantes = pd.DataFrame({
        'Faltantes (Qtd)': qtd_faltantes,
        'Percentual (%)': perc_faltantes
    })
    
    # 4. Filtra apenas colunas que têm dados faltando e ordena
    tabela_filtrada = tabela_faltantes[tabela_faltantes['Faltantes (Qtd)'] > 0]
    tabela_filtrada = tabela_filtrada.sort_values(by='Percentual (%)', ascending=False)
    
    print(tabela_filtrada)

# Execução independente
if __name__ == "__main__":
    analisar_faltantes("AirbnbRJRentals.csv")



--- 2. DIAGNÓSTICO DE DADOS FALTANTES ---
                   Faltantes (Qtd)  Percentual (%)
reviews_per_month             8518       30.097876
last_review                   8518       30.097876
name                            29        0.102470
host_name                        3        0.010600


2. Diagnóstico de qualidade
2.3 Duplicados, categorias inconsistentes, valores inválidos
Aqui verificamos preços zerados, linhas duplicadas e alguma inconsistências

In [3]:
import pandas as pd

def validar_regras_negocio(caminho_arquivo):
    print("\n--- 3. DUPLICADOS E INCONSISTÊNCIAS ---")
    
    # 1. Carrega o CSV direto neste script
    df = pd.read_csv(caminho_arquivo)
    
    # 2. Checagem de linhas duplicadas
    duplicados = df.duplicated().sum()
    print(f"Linhas perfeitamente duplicadas: {duplicados}")
    
    # 3. Categorias inconsistentes (verificando se há tipos de quartos estranhos)
    print("\nDistribuição de Tipos de Quarto (room_type):")
    print(df['room_type'].value_counts())
    
    # 4. Valores inválidos baseados na lógica do mundo real
    precos_invalidos = (df['price'] <= 0).sum()
    noites_invalidas = (df['minimum_nights'] <= 0).sum()
    
    print(f"\nAnúncios com preço R$ 0 ou negativo: {precos_invalidos}")
    print(f"Anúncios com noites mínimas zeradas ou negativas: {noites_invalidas}")

# Execução independente
if __name__ == "__main__":
    validar_regras_negocio("AirbnbRJRentals.csv")



--- 3. DUPLICADOS E INCONSISTÊNCIAS ---
Linhas perfeitamente duplicadas: 0

Distribuição de Tipos de Quarto (room_type):
room_type
Entire home/apt    22093
Private room        5685
Shared room          470
Hotel room            53
Name: count, dtype: int64

Anúncios com preço R$ 0 ou negativo: 6
Anúncios com noites mínimas zeradas ou negativas: 0


2. Diagnóstico de qualidade
2.4 Identificação de outliers com método justificado (IQR ou z-score)
Optamos por utilizar o IQR, porque é melhor para analisar preços distorcidos. O jeito que é feito o cálculo no Z-score pode distorcer o cálculo (média e desvio padrão). O IQR funciona com qualquer distribuição também, o z-score é feito para analisar distribuição normal (gaussiana).

In [4]:
import pandas as pd

def identificar_outliers_iqr(caminho_arquivo, coluna):
    print(f"\n--- 4. IDENTIFICAÇÃO DE OUTLIERS: Coluna '{coluna}' ---")
    
    # 1. Carrega o CSV direto neste script
    df = pd.read_csv(caminho_arquivo)
    
    # 2. Cálculo dos Quartis (25% e 75%)
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    
    # 3. Cálculo do Intervalo Interquartil (IQR)
    IQR = Q3 - Q1
    
    # 4. Definição dos limites estatísticos
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # 5. Filtragem dos registros que estão fora dos limites
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]
    perc_outliers = (len(outliers) / len(df)) * 100
    
    print("Método Justificado: IQR (Intervalo Interquartil)")
    print(f"Q1 (25%): R$ {Q1:.2f} | Q3 (75%): R$ {Q3:.2f}")
    print(f"Limite Superior de Corte: R$ {limite_superior:.2f}")
    print(f"Total de Outliers encontrados: {len(outliers)} anúncios ({perc_outliers:.2f}% da base)")
    
    # 6. Retorna a base limpa sem os outliers para análises futuras
    df_sem_outliers = df[(df[coluna] >= limite_inferior) & (df[coluna] <= limite_superior)]
    return df_sem_outliers

# Execução independente focada na coluna de preço
if __name__ == "__main__":
    df_limpo = identificar_outliers_iqr("AirbnbRJRentals.csv", "price")



--- 4. IDENTIFICAÇÃO DE OUTLIERS: Coluna 'price' ---
Método Justificado: IQR (Intervalo Interquartil)
Q1 (25%): R$ 317.00 | Q3 (75%): R$ 1071.00
Limite Superior de Corte: R$ 2202.00
Total de Outliers encontrados: 2450 anúncios (8.66% da base)


3. Limpeza e transformação
Após carregar a tabela no Dataframe tiramos os outliers (código 3.1_analise_preco.py presente na pasta notebooks do branche main), após pesquisas e recomendações da IA utilizamos o IQR pois é mais robusto para esse caso, médias e desvios padrão dependendo do preços, caso o anfitrião erre no preço ou coloque de uma hora pra outra um preço muito alto pode atrapalhar a análise.

In [5]:
import pandas as pd

def analisar_preco_localizacao_texto(caminho_arquivo):
    # 1. Carregar os dados
    df = pd.read_csv(caminho_arquivo)
    
    # 2. Limpeza de Outliers (Filtro estatístico pelo método IQR)
    Q1 = df['price'].quantile(0.25)
    Q3 = df['price'].quantile(0.75)
    IQR = Q3 - Q1
    limite_superior = Q3 + 1.5 * IQR
    
    # Filtra preços zerados e valores absurdos
    df_limpo = df[(df['price'] > 0) & (df['price'] <= limite_superior)]
    
    # 3. Métricas Gerais
    print("=========================================")
    print("       RESUMO DE PREÇOS (RJ)             ")
    print("=========================================")
    print(f"Preço Médio na cidade:   R$ {df_limpo['price'].mean():.2f}")
    print(f"Preço Mediano na cidade: R$ {df_limpo['price'].median():.2f}")
    
    # 4. Agrupamento por Bairro (Filtrando bairros com pelo menos 10 anúncios)
    bairros_stats = df_limpo.groupby('neighbourhood')['price'].agg(['mean', 'median', 'count'])
    bairros_validos = bairros_stats[bairros_stats['count'] >= 10]
    
    # Ordenação
    top_10_caros = bairros_validos.sort_values(by='mean', ascending=False).head(10)
    top_10_baratos = bairros_validos.sort_values(by='mean', ascending=True).head(10)
    
    # 5. Exibição Tabular
    print("\n=========================================")
    print("   TOP 10 BAIRROS MAIS CAROS (Média)     ")
    print("=========================================")
    # Renomeando as colunas para ficar mais amigável no terminal
    top_10_caros_formatado = top_10_caros.rename(columns={'mean': 'Preço Médio (R$)', 'median': 'Mediana (R$)', 'count': 'Qtd Anúncios'})
    print(top_10_caros_formatado.round(2))
    
    print("\n=========================================")
    print("  TOP 10 BAIRROS MAIS BARATOS (Média)    ")
    print("=========================================")
    top_10_baratos_formatado = top_10_baratos.rename(columns={'mean': 'Preço Médio (R$)', 'median': 'Mediana (R$)', 'count': 'Qtd Anúncios'})
    print(top_10_baratos_formatado.round(2))

# Execução do script
if __name__ == "__main__":
    analisar_preco_localizacao_texto("AirbnbRJRentals.csv")


       RESUMO DE PREÇOS (RJ)             
Preço Médio na cidade:   R$ 666.33
Preço Mediano na cidade: R$ 548.00

   TOP 10 BAIRROS MAIS CAROS (Média)     
                          Preço Médio (R$)  Mediana (R$)  Qtd Anúncios
neighbourhood                                                         
Leblon                              979.63         919.5          1218
Ipanema                             927.25         865.0          2360
Lagoa                               859.20         785.0           202
São Conrado                         833.30         786.0           155
Joá                                 807.87         639.5            46
Barra da Tijuca                     803.58         708.5          2514
Gávea                               756.80         629.0           171
Alto da Boa Vista                   747.17         750.0            29
Copacabana                          708.22         614.0          7711
Recreio dos Bandeirantes            682.42         562.0        